# Module 3 — Crop Performance Index & Yield (maize, GHA)
**CPI** = multi-stress multiplicative stacking (water × heat × vegetation), stage-weighted; then **yield (t/ha) = CPI/100 × Ym** and **production** = yield × area (250 m pixel = 6.25 ha).

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

### Stage 0 · Runtime

**What runs.** Installs the Earth Engine Python client and `geemap` into the Colab runtime. Nothing is
computed here.

**Expected output.** One line, `installed.`, after 30 to 60 s on a cold runtime. Pip warnings about
dependency resolution are normal and can be ignored.

**If it fails.** Re-run the cell. A repeated failure usually means the runtime lost its network
connection; use *Runtime → Restart session* and start again.

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine sign-in

**What runs.** Connects to Earth Engine under the cloud project `PROJECT`. On a fresh runtime a
browser prompt appears; approve it with the Google account that has Earth Engine access.

**Expected output.** `EE ready: ok` within a few seconds. Anything else means the sign-in did not
complete.

**Which project to use.** Compute is identical across projects, but the **export queue is per
project**. `ee-manzikye` has stalled with tasks sitting in READY for hours. If you are going to
export, set `PROJECT = "indigo-proxy-484220-q8"` before running.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Pipeline code on Drive

**What runs.** Mounts Google Drive and puts `/content/drive/MyDrive/planting_pipeline` on the Python
path, so `from src import ...` resolves to the pipeline modules rather than to anything installed by
pip.

**Expected output.** `Mounted at /content/drive` followed by
`pipeline on path: /content/drive/MyDrive/planting_pipeline`.

**If you get an `AssertionError`.** The folder is not where the cell expects it. Either upload the
whole `planting_pipeline` folder to the top level of My Drive, or edit `PIPE_DIR` to the real path.
The folder must contain `run.py`, `src/` and `config/`.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Stage 0d · Run configuration

**What you choose here.**

| Variable | Meaning |
|---|---|
| `COUNTRY`, `SEASON` | select a row of `config/season_calendar.csv`; this fixes the season window and the crop calendar |
| `YEAR` | the season's planting year. A season that crosses new year (short rains, Deyr) is still keyed by its planting year |
| `S1_ORBIT` | Sentinel-1 orbit. `ASCENDING` over Kenya, because Sentinel-1B failed in 2022 and descending coverage is sparse |
| `aoi` | the whole country, from the GAUL level-0 boundary |
| `aoi_run` | the area actually computed. It ships as a **test box**, 34.4 to 37.8 E and 1.2 S to 1.2 N, about 380 by 265 km over western and central Kenya |

**Time is counted in dekads, not dates.** A dekad is a third of a month, numbered 1 to 36 through the
year: dekad 1 is 1 to 10 January, dekad 9 is 21 to 31 March, dekad 36 is 21 to 31 December. Days 21 to
the month end are one dekad, so a dekad is 8, 9, 10 or 11 days long. `utils.dekad_label(9)` prints
`9·Mar`. Where a season crosses the new year the code uses a **global dekad** `gd` running 1 to 72,
which is the dekad of `YEAR` for 1 to 36 and of `YEAR + 1` for 37 to 72.

**Season windows this notebook can use.**

| Country · season | SOS detection window | Dekads |
|---|---|---|
| Kenya · Long rains | Mar-d3 to May-d3 | 9 to 15 |
| Kenya · Short rains | Oct-d1 to Nov-d3 | 28 to 33 |
| Ethiopia · Meher | Apr-d2 to Jun-d3 | 11 to 18 |

**Expected output.** One line, for example `Kenya · Long rains · 2024 · S1 ASCENDING`.

**Before you switch to the whole country.** Replace `aoi_run` with `aoi` only when the test box has
run cleanly. The country is roughly ten times the area, and Sentinel-2 and Sentinel-1 compositing
scales with it. Expect minutes to become tens of minutes, and expect `getInfo()` calls to time out;
at country scale use `ee.batch.Export` instead of reading results back into the notebook.

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting (onset anchor)

### Stage 1 · Planting dekad

**What this stage does.** It estimates, for every maize pixel, the dekad the crop was planted. Every
later module is anchored on this number, so an error here propagates into the water balance, the CPI
and the yield. Two different methods run, chosen by season.

**Main seasons: cue-fusion green-up.** Optical greenness is combined with radar so that cloud does not
leave holes. For each dekad a fused greenness proxy is built,

$$G_t=\tfrac{1}{2}\Big[\mathrm{unit}(\mathrm{NDRE}_t;0,0.7)+\mathrm{unit}(\mathrm{FPAR}_t;0,0.9)\Big],
\qquad G_t \leftarrow \mathrm{unit}(\mathrm{RVI}_t;0.1,0.8)\ \text{where optical is missing,}$$

where $\mathrm{unit}(x;a,b)$ rescales $x$ from $[a,b]$ to $[0,1]$. NDRE is the Sentinel-2 red-edge
index, FPAR is MODIS MCD15A3H, and RVI is the Sentinel-1 radar vegetation index, which rises with
canopy and is unaffected by cloud.

Start of season is the first dekad in the window at which greenness crosses a quarter of the season's
own amplitude and is still rising:

$$G_{\text{thr}}=G_{\min}+0.25\,(G_{\max}-G_{\min}),\qquad
\mathrm{SOS}=\min\{\,t:\ G_t\ge G_{\text{thr}}\ \wedge\ G_{t+1}-G_t\ge 0\ \wedge\ |t-\mathrm{SOS}_{\mathrm{LTN}}|\le 2\,\}$$

The last condition keeps the answer within two dekads of the climatological onset, which rejects weed
flushes and a second green-up. It is applied only where a climatology exists, so a sparse second-season
normal cannot reject every pixel.

Planting precedes visible green-up, so the detected SOS is shifted back by the crop's emergence lag:

$$\text{planting dekad} = \mathrm{SOS} - 2 \quad \text{(maize; wheat and teff use 1).}$$

**Short rains: rainfall onset.** Green-up detection is unreliable in the short rains, so the FEWS NET
rule is used instead. Onset is the first dekad with

$$P_t \ge 25\ \mathrm{mm}\quad\text{and}\quad P_{t+1}+P_{t+2}\ge 20\ \mathrm{mm}
\quad\text{and}\quad P_t/ET_{0,t}\ge 0.5 .$$

The first two conditions are the classic 25/20 mm rule; the third is an agroclimatic gate that asks
whether the rain was large relative to evaporative demand.

**Expected output.** A single line, `planting dekad computed for <country> <season>`. Nothing is
evaluated yet: Earth Engine is lazy, so errors in this cell often only surface at the next one, where
a number is actually requested.

**Expected values.** The result must fall inside the SOS window of the table above, minus the
emergence offset. For Kenya long rains 2024 the modal planting dekad is **8** (11 to 20 March), with
the 10th to 90th percentile of the 253 constituencies spanning dekads **7 to 9**. A modal dekad
outside 6 to 11 for that season means the fusion locked onto the wrong green-up.

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

### Stage 2 · Crop performance index and yield

**What this stage does.** It converts three separate stresses into one relative-yield index, then into a
yield in tonnes per hectare.

**The stacking.** Following AquaCrop and the crop-model convention, each hazard is an independent
fractional yield reduction and they multiply:

$$\frac{Y_a}{Y_m}=(1-S_{\text{water}})(1-S_{\text{heat}})(1-S_{\text{veg}}),
\qquad \mathrm{CPI}=100\,\frac{Y_a}{Y_m}.$$

Multiplying, rather than adding, means two moderate stresses compound but neither alone can take the
index to zero.

**Water stress, FAO-33.** Built from the per-stage actual and required evapotranspiration of the water
balance, weighted by the FAO-33 yield-response factors $K_y$:

$$S_{\text{water}}=\sum_{s\in\{\text{veg},\text{flo},\text{grf}\}}
K_{y,s}\left(1-\frac{AET_s}{WR_s}\right),
\qquad K_y = 0.4,\ 1.5,\ 0.5,$$

clamped to $[0,1]$. Flowering carries three times the weight of grain filling, which is the whole point
of running the balance stage by stage rather than over the season.

**Heat stress.** Accumulated heat-degree-dekads above a cap, over the flowering window only, because the
damage mechanism is pollen sterility:

$$S_{\text{heat}}=\min\Big(1,\ 0.06 \sum_{t\in\text{flowering}} \max(T_{\max,t}-33,\ 0)\Big),$$

with $T_{\max}$ the dekad-mean daily maximum from ERA5-Land.

**Expect this term to be zero in Kenya, and treat that as correct.** The cap is a dekad-mean, and
measured dekad-mean flowering $T_{\max}$ peaks at 22.9, 25.9 and 29.3 °C across Kenya's three season
regimes, so 33 °C cannot be reached in two of them. The term is live for lowland and Sahelian seasons.
If you are testing the threshold, pass `tcap=` and `k=` to `s_heat` rather than editing the module.

**Vegetation stress.** A deliberately down-weighted confirmation from the satellite, not a driver:

$$S_{\text{veg}}=0.4\,(1-\mathrm{VCI}),\qquad
\mathrm{VCI}=\frac{\mathrm{NDVI}_{\text{peak}}-\mathrm{NDVI}_{\min}}{\mathrm{NDVI}_{\max}-\mathrm{NDVI}_{\min}},$$

over the 2003 to 2023 climatology. The weight of 0.4 caps its contribution at a 40 % yield reduction,
because a vegetation index confirms a stress but does not measure yield. Setting `VEG_INDEX=fpar` swaps
VCI for the standardised FPAR anomaly used by JRC ASAP, $S_{\text{veg}}=0.4\,\mathrm{clamp}(-z/2,0,1)$.

**Yield and production.**

$$Y_a = \frac{\mathrm{CPI}}{100}\times Y_m,\qquad
\text{production (t)} = Y_a \times 6.25\ \text{ha per 250 m pixel}.$$

**$Y_m$ is calibrated, not a textbook potential.** `CPI.ym_for(COUNTRY, SEASON)` returns the ceiling
fitted to HarvestStat sub-national yields, target = the median over the available years, least squares
through the origin, tested on a 70/30 split repeated 200 times:

| Country · season | $Y_m$ t/ha | Units | Held-out MAE, calibrated vs default | $r$ |
|---|---|---|---|---|
| Kenya · Long rains | **2.34** | 45 | 0.68 vs 2.26 | 0.57 |
| Kenya · Short rains | **1.44** | 44 | 0.39 vs 1.94 | 0.36 |
| Ethiopia · Meher | **4.14** | 76 | 0.71 vs 1.38 | 0.64 |
| Rwanda · Season A | **2.61** | 30 | 0.37 vs 2.75 | −0.07 |
| Burundi · Season A | **1.88** | 16 | 0.71 vs 3.57 | 0.28 |
| Somalia · Gu | **1.02** | 18 | 0.24 vs 1.58 | 0.34 |
| Uganda · 1st rains | 2.34 | 74 | 1.24 vs 2.90 | −0.14 (provisional) |

Tanzania and South Sudan have no HarvestStat maize yields and fall back to the uncalibrated 6.0 t/ha,
which every country that could be tested shows to be several times too high. Where $r$ is near zero the
ceiling fixes the **level** only: use the map for national and seasonal totals, not to rank districts.

**Expected output.** One line ending in the AOI total production in tonnes. Two sanity checks:

* CPI over maize should mostly sit between **55 and 85** in a normal Kenyan long rains. That is what a
  reported yield of about 1.3 to 1.8 t/ha implies against a 2.34 t/ha ceiling.
* Mean yield should land near **1.5 t/ha** for Kenya long rains, near **3 t/ha** for Ethiopia Meher.
  A mean above 4 t/ha for Kenya means `ym_for` fell through to a default, which happens when `COUNTRY`
  or `SEASON` is spelled differently from the calendar file.

**A CPI of 100 is not a good season, it is a missing stress.** If the whole map reads near 100, check
that the water balance ran: an empty `staged` dictionary makes every stress zero.

In [ ]:
# --- CPI + yield ---
from src.wrsi_waterbalance import run_wrsi_staged
from src import cpi as CPI, soil as SOIL
mz=kc['maize']; d_veg=mz['L_ini']+mz['L_dev']; d_flo=d_veg+mz['L_mid']; lgp=mz['LGP_dekads']
YM=4.5 if SEASON=='Short rains' else 6.0
whc=SOIL.get_whc(ee,aoi_run,soil,root_depth_cm=int(mz.get('root_depth_m',1.0)*100))
staged=run_wrsi_staged(ee,aoi_run,YEAR,planting,'maize',kc,soil,ss,se,whc_img=whc)
Sw=CPI.s_water(ee,staged); Sh=CPI.s_heat(ee,aoi_run,YEAR,planting,d_veg,d_flo,ss,se); Sv=CPI.s_veg(ee,aoi_run,YEAR,ss,se)
cpi_img,yld=CPI.cpi(ee,Sw,Sh,Sv,ym=YM)
PIXEL_HA=6.25; production=yld.multiply(PIXEL_HA)
tot=yld.updateMask(mask).multiply(PIXEL_HA).reduceRegion(ee.Reducer.sum(),aoi_run,250,maxPixels=int(1e13)).get('yield_tha').getInfo()
print('CPI + yield computed · AOI total production (t, indicative):', round(tot))

### Stage 3 · Map

CPI is drawn 0 to 100 and yield 0 to 6 t/ha on the same map so they can be toggled against each other.
They carry identical spatial pattern by construction, because yield is CPI times a constant; the second
layer exists to put the pattern in units a user recognises.

**What to check.** The yield layer should have no values above the country's $Y_m$. If it does, `ym`
was passed as an image with the highland split enabled, which is switched off under the typical-year
calibration.

In [ ]:
M=new_map()
ee_layer(M, cpi_img.updateMask(mask).clip(aoi_run), {'min':0,'max':100,'palette':['a50026','fee08b','1a9850']}, 'CPI (0-100)')
ee_layer(M, yld.updateMask(mask).clip(aoi_run), {'min':0,'max':6,'palette':['ffffcc','78c679','006837']}, 'Yield (t/ha)')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

*Ym is a **reference** potential — calibrate against observed yields (KALRO / HarvestStat). See `CPI_METHODOLOGY`, `YIELD_ESTIMATION_METHODOLOGY`.*